# Experimentos: forma de trayectoria y embeddings

El notebook 05 mostró que, con las tasas crudas, los perfiles ganadores son niveles de
participación (35 / 49 / 62 / 75%) con brecha y tendencia casi iguales entre si.
Aquí se prueba agrupar por la **forma** de la trayectoria, quitando el nivel de cada celda,
y se agregan **UMAP** y **t-SNE** como reducción previa al clustering.

Cada configuración se evalua con dos lentes:
- **nivel:** que tan bien separa y explica el promedio de participación.
- **forma:** que tan bien separa y explica la brecha presidencial y la tendencia.

## 1. Configuración

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.stats import chi2_contingency
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 2. Parametros y rutas

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"

ELECCIONES = ["P2009", "P2012", "P2015", "P2018", "P2021", "P2024"]
CORTES_EDAD = [17, 24, 29, 39, 49, 59, 69, 79, 200]
ETIQUETAS_EDAD = ["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]

K_RANGE = range(3, 9)
K_VALIDO = range(4, 7)  # convocatoria pide 4-6 perfiles
MIN_PCT_PERFIL = 5.0
MAX_PCT_RUIDO = 10.0

N_FIT = 60_000
N_SUB = 8_000
N_EVAL = 10_000
N_SIL = 5_000
N_FINAL = 200_000
N_BOOT = 3
ARI_ESTABLE = 0.8
RANDOM_STATE = 42

## 3. Carga de datos

### 3.1 Celdas, peso y PJ 2025

Mismo procedimiento que el notebook 05: celdas del notebook 03, peso = SV + NV de 2024
y solo casos completos.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet")
celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)

fed24 = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["EDOCVE", "SECCIÓN", "SEXO", "EDAD", "AELEC", "SV", "NV"],
    filters=[("AELEC", "==", 2024)],
)
fed24["GRUPO_EDAD"] = pd.cut(fed24["EDAD"], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD)
fed24["VOTANTES"] = fed24["SV"] + fed24["NV"]
pesos = (
    fed24.groupby(["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], observed=True)["VOTANTES"]
    .sum()
    .reset_index()
    .rename(columns={"VOTANTES": "peso"})
)
pesos["GRUPO_EDAD"] = pesos["GRUPO_EDAD"].astype(str)
del fed24

secciones = pd.read_parquet(
    DIR_PREPROCESSED / "secciones_historial.parquet", columns=["EDOCVE", "SECCIÓN", "P_PJ2025"]
)
celdas = (
    celdas.merge(pesos, on=["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], how="left")
    .merge(secciones, on=["EDOCVE", "SECCIÓN"], how="left")
    .dropna(subset=["peso", "P_PJ2025"])
    .reset_index(drop=True)
)
del pesos, secciones
celdas["MUJER"] = (celdas["SEXO"] == 1).astype(int)
print(f"Celdas completas: {len(celdas):,}  peso: {celdas['peso'].sum():,.0f}")

### 3.2 Forma de la trayectoria

- **F (centrada):** tasa de cada elección menos el promedio de la celda. Conserva la amplitud
  de las oscilaciones (en puntos) pero elimina el nivel.
- **Z (estandarizada por fila):** la forma centrada dividida entre su desviación. Solo queda el
  patrón (que elecciones están arriba o abajo), sin nivel ni amplitud.

In [ ]:
P = celdas[ELECCIONES].to_numpy(dtype=float)
forma = P - P.mean(axis=1, keepdims=True)
amplitud = forma.std(axis=1, keepdims=True)
forma_z = forma / np.where(amplitud < 1e-6, 1.0, amplitud)

FORMA = [f"F{e[1:]}" for e in ELECCIONES]
FORMA_Z = [f"Z{e[1:]}" for e in ELECCIONES]
celdas[FORMA] = forma
celdas[FORMA_Z] = forma_z
del P, forma, forma_z

print(f"Diferencia máxima entre promedio y media de la fila: "
      f"{(celdas['promedio'] - celdas[ELECCIONES].mean(axis=1)).abs().max():.4f}")
celdas[FORMA].describe().round(2)

## 4. Diseño del experimento

### 4.1 Conjuntos de variables

- **A_base** y **C_trayectoria:** referencias (modelo del notebook 04 y ganador del 05).
- **S_forma:** forma centrada.
- **S_forma_pj:** forma centrada más PJ 2025.
- **S_forma_z:** forma estandarizada por fila.

In [ ]:
CONJUNTOS = {
    "A_base": ["promedio", "brecha_pres", "tendencia"],
    "C_trayectoria": ELECCIONES,
    "S_forma": FORMA,
    "S_forma_pj": FORMA + ["P_PJ2025"],
    "S_forma_z": FORMA_Z,
}
PREPROCESOS = ["estándar", "estandar_pca"]

COMBOS = [
    (c, pre) for c, cols in CONJUNTOS.items() for pre in PREPROCESOS
    if not (pre == "estandar_pca" and len(cols) <= 3)
]
print(f"Combinaciones variables x preprocesamiento: {len(COMBOS)}")

### 4.2 Muestras ponderadas

Igual que en el notebook 05: eval (10k) fija para métricas, fit (60k) para KMeans y GMM
sin embedding, sub (8k) para todo lo demás. Probabilidad proporcional al peso.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
p_all = celdas["peso"].to_numpy() / celdas["peso"].sum()
idx_eval = rng.choice(len(celdas), size=N_EVAL, replace=False, p=p_all)
resto = np.setdiff1d(np.arange(len(celdas)), idx_eval)
p_resto = p_all[resto] / p_all[resto].sum()


def muestra_ajuste(seed, n=N_FIT):
    idx = np.random.default_rng(seed).choice(resto, size=n, replace=False, p=p_resto)
    return idx, idx[:N_SUB]


idx_fit, idx_sub = muestra_ajuste(RANDOM_STATE)
print(f"eval={len(idx_eval):,}  fit={len(idx_fit):,}  sub={len(idx_sub):,}")

### 4.3 Espacios de referencia

- **nivel:** 6 tasas federales y PJ 2025 estandarizadas (el mismo del notebook 05).
- **forma:** las 6 tasas centradas por fila, estandarizadas.

Ambos se calculan siempre en el espacio original, nunca en el embedding: UMAP y t-SNE
exageran la separación entre grupos y harían ver bien cualquier particion.

In [ ]:
def espacio(cols):
    sc = StandardScaler().fit(celdas.loc[idx_fit, cols])
    return sc.transform(celdas.loc[idx_eval, cols])


X_nivel = espacio(ELECCIONES + ["P_PJ2025"])
X_forma = espacio(FORMA)
ev = celdas.loc[idx_eval]
print(f"nivel {X_nivel.shape}  forma {X_forma.shape}")

### 4.4 Embeddings

| Embedding | Hiperparametros |
|---|---|
| ninguno | - |
| UMAP | n_neighbors (15, 50), min_dist (0.0, 0.1), dimensiones (2, 5) |
| t-SNE | perplejidad (30, 100), 2 dimensiones |

Se ajustan sobre la muestra sub. t-SNE no tiene `transform`, por eso para ambos se agrupa
en el embedding y las etiquetas se extienden a otras filas con KNN en el espacio original.

In [ ]:
EMBEDDINGS = [("ninguno", {})]
EMBEDDINGS += [("umap", {"nn": nn, "md": d, "dim": dim})
               for nn in [15, 50] for d in [0.0, 0.1] for dim in [2, 5]]
EMBEDDINGS += [("tsne", {"perp": p}) for p in [30, 100]]


def embeber(emb, prm, Z_sub, seed):
    if emb == "umap":
        return umap.UMAP(n_neighbors=prm["nn"], min_dist=prm["md"], n_components=prm["dim"],
                         random_state=seed).fit_transform(Z_sub)
    return TSNE(n_components=2, perplexity=prm["perp"], init="pca",
                random_state=seed).fit_transform(Z_sub)


def texto(prm):
    return ", ".join(f"{k}={v}" for k, v in prm.items())


print(f"Embeddings: {len(EMBEDDINGS)}")

### 4.5 Algoritmos de clustering

Los que mostraron estabilidad en el notebook 05 (KMeans, GMM, ward) más HDBSCAN, que suele
funcionar mejor sobre UMAP porque el embedding concentra la densidad.

In [ ]:
MALLA = []
for k in K_RANGE:
    MALLA += [("kmeans", {"k": k}), ("gmm", {"k": k}), ("ward", {"k": k})]
MALLA += [("hdbscan", {"mcs": mcs, "ms": ms}) for mcs in [80, 200, 400] for ms in [5, 25]]


def agrupar(algo, prm, X, seed, cache):
    if algo == "kmeans":
        return KMeans(prm["k"], n_init=4, random_state=seed).fit_predict(X)
    if algo == "gmm":
        return GaussianMixture(prm["k"], n_init=2, random_state=seed).fit(X).predict(X)
    if algo == "ward":
        if "ward" not in cache:
            cache["ward"] = linkage(X, method="ward")
        return fcluster(cache["ward"], prm["k"], criterion="maxclust") - 1
    return HDBSCAN(min_cluster_size=prm["mcs"], min_samples=prm["ms"]).fit_predict(X)


def ajustar(emb, algo, prm, Z_fit, Z_sub, E_sub, seed, cache, w=None, n_init=None):
    if emb == "ninguno" and algo == "kmeans":
        return KMeans(prm["k"], n_init=n_init or 4, random_state=seed).fit(Z_fit, sample_weight=w)
    if emb == "ninguno" and algo == "gmm":
        return GaussianMixture(prm["k"], n_init=n_init or 2, random_state=seed).fit(Z_fit)
    X = Z_sub if emb == "ninguno" else E_sub
    lab = agrupar(algo, prm, X, seed, cache)
    return KNeighborsClassifier(n_neighbors=5).fit(Z_sub, lab)


print(f"Configuraciones de clustering por embedding: {len(MALLA)}")
print(f"Total: {len(COMBOS) * len(EMBEDDINGS) * len(MALLA):,}")

### 4.6 Métricas

- **sil, db:** en el espacio propio preprocesado (no en el embedding).
- **sil_nivel, r2_prom:** separación en el espacio de nivel y varianza explicada del promedio.
- **sil_forma, r2_forma:** separación en el espacio de forma y varianza explicada promedio
  de brecha presidencial y tendencia.
- **v_edad, v_sexo:** V de Cramer con grupo de edad y sexo (informativas).

In [ ]:
EN_SIL = np.zeros(N_EVAL, dtype=bool)
EN_SIL[np.random.default_rng(RANDOM_STATE).permutation(N_EVAL)[:N_SIL]] = True


def silhouette_seguro(Z, lab):
    return silhouette_score(Z, lab) if len(np.unique(lab)) > 1 else np.nan


def cramers_v(a, b):
    tabla = pd.crosstab(a, b).to_numpy()
    if min(tabla.shape) < 2:
        return 0.0
    chi2 = chi2_contingency(tabla, correction=False)[0]
    return float(np.sqrt(chi2 / (tabla.sum() * (min(tabla.shape) - 1))))


def eta2(y, labels):
    medias = pd.Series(y).groupby(labels).transform("mean").to_numpy()
    return float(((medias - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def evaluar(labels, Z_eval):
    mask = labels >= 0
    m_sil = mask & EN_SIL
    lab = labels[mask]
    tamanos = pd.Series(lab).value_counts() / len(labels) * 100
    res = {"k_eff": len(tamanos), "pct_ruido": (1 - mask.mean()) * 100,
           "min_pct": tamanos.min() if len(tamanos) else 0.0}
    if len(tamanos) < 2:
        return res
    e = ev[mask]
    res.update({
        "sil": silhouette_seguro(Z_eval[m_sil], labels[m_sil]),
        "db": davies_bouldin_score(Z_eval[mask], lab),
        "sil_nivel": silhouette_seguro(X_nivel[m_sil], labels[m_sil]),
        "r2_prom": eta2(e["promedio"].to_numpy(), lab),
        "sil_forma": silhouette_seguro(X_forma[m_sil], labels[m_sil]),
        "r2_forma": (eta2(e["brecha_pres"].to_numpy(), lab) + eta2(e["tendencia"].to_numpy(), lab)) / 2,
        "v_edad": cramers_v(lab, e["GRUPO_EDAD"].to_numpy()),
        "v_sexo": cramers_v(lab, e["SEXO"].to_numpy()),
    })
    return res

## 5. Búsqueda en malla

Para cada combinacion se ajusta el preprocesamiento, se calculan los embeddings sobre sub
y se corren todas las configuraciones de clustering. Los embeddings se guardan para graficarlos.

In [ ]:
CONFIGS, ETIQUETAS, EMB_SUB, registros = {}, {}, {}, []
t_inicio = time.perf_counter()

for conjunto, prep in tqdm(COMBOS, desc="combinaciones"):
    X = celdas[CONJUNTOS[conjunto]].to_numpy(dtype=float)
    tr = (StandardScaler() if prep == "estándar"
          else make_pipeline(StandardScaler(), PCA(n_components=0.9, random_state=RANDOM_STATE)))
    tr.fit(X[idx_fit])
    Z_fit, Z_eval = tr.transform(X[idx_fit]), tr.transform(X[idx_eval])
    Z_sub = Z_fit[:N_SUB]

    for emb, eprm in EMBEDDINGS:
        t0 = time.perf_counter()
        E_sub = None if emb == "ninguno" else embeber(emb, eprm, Z_sub, RANDOM_STATE)
        t_emb = time.perf_counter() - t0
        EMB_SUB[(conjunto, prep, emb, texto(eprm))] = E_sub
        cache = {}
        for algo, prm in MALLA:
            cid = len(CONFIGS)
            CONFIGS[cid] = (conjunto, prep, emb, eprm, algo, prm)
            modelo = ajustar(emb, algo, prm, Z_fit, Z_sub, E_sub, RANDOM_STATE, cache)
            labels = modelo.predict(Z_eval)
            ETIQUETAS[cid] = labels
            registros.append({
                "id": cid, "conjunto": conjunto, "prep": prep, "embedding": emb,
                "emb_params": texto(eprm), "algoritmo": algo, "params": texto(prm),
                **evaluar(labels, Z_eval), "seg_embedding": t_emb,
            })

res = pd.DataFrame(registros)
print(f"{len(res):,} configuraciones en {(time.perf_counter() - t_inicio) / 60:.1f} min")

### 5.1 Validez y puntajes

Mismo filtro que el notebook 05. Dos puntajes entre las validas (promedio de rangos percentiles):

- **score_nivel:** sil, sil_nivel, r2_prom.
- **score_forma:** sil, sil_forma, r2_forma.

In [ ]:
def motivo(r):
    if r.k_eff not in K_VALIDO:
        return "k fuera de 4-6"
    if r.pct_ruido > MAX_PCT_RUIDO:
        return "ruido > 10%"
    if r.min_pct < MIN_PCT_PERFIL:
        return "perfil < 5%"
    return "valida"


res["motivo"] = res.apply(motivo, axis=1)
validas = res["motivo"] == "valida"
for nombre, cols in [("score_nivel", ["sil", "sil_nivel", "r2_prom"]),
                     ("score_forma", ["sil", "sil_forma", "r2_forma"])]:
    res[nombre] = np.nan
    res.loc[validas, nombre] = res.loc[validas, cols].rank(pct=True).mean(axis=1)
print(f"Validas: {validas.sum():,} de {len(res):,}")
pd.crosstab([res["embedding"], res["algoritmo"]], res["motivo"])

### 5.2 Métricas promedio por conjunto y embedding

Solo configuraciones validas.

In [ ]:
METRICAS = ["sil", "sil_nivel", "r2_prom", "sil_forma", "r2_forma", "v_edad", "v_sexo"]
res[validas].groupby(["conjunto", "embedding"])[METRICAS].mean().round(3)

### 5.3 Nivel vs forma

Cada punto es una configuración valida. Los conjuntos de forma deberían moverse hacia arriba
(mejor forma) a costa de ir a la izquierda (peor nivel).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for conjunto, d in res[validas].groupby("conjunto"):
    ax.scatter(d["r2_prom"], d["r2_forma"], s=12, alpha=0.6, label=conjunto)
ax.set_xlabel("r2_prom (varianza del promedio explicada)")
ax.set_ylabel("r2_forma (varianza de brecha y tendencia explicada)")
ax.set_title("Configuraciones validas: nivel vs forma")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Embeddings de la forma

UMAP (n_neighbors=15, min_dist=0.1, 2D) y t-SNE (perplejidad 30) de S_forma, coloreados por
brecha presidencial. Si hubiera grupos naturales se verian islas separadas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
color = celdas.loc[idx_sub, "brecha_pres"].clip(-10, 40)
for ax, (emb, eprm) in zip(axes, [("umap", "nn=15, md=0.1, dim=2"), ("tsne", "perp=30")]):
    E = EMB_SUB[("S_forma", "estándar", emb, eprm)]
    sc = ax.scatter(E[:, 0], E[:, 1], c=color, s=2, cmap="viridis")
    ax.set_title(f"{emb.upper()} ({eprm})")
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(sc, ax=axes, label="brecha_pres (recortada a -10..40)")
plt.show()

### 5.5 Top 15 por cada puntaje

In [ ]:
COLS_TABLA = ["id", "conjunto", "prep", "embedding", "emb_params", "algoritmo", "params",
              "k_eff", "min_pct", "sil", "sil_nivel", "r2_prom", "sil_forma", "r2_forma",
              "v_edad", "v_sexo", "score_nivel", "score_forma"]
res[validas].nlargest(15, "score_forma")[COLS_TABLA].round(3)

In [ ]:
res[validas].nlargest(15, "score_nivel")[COLS_TABLA].round(3)

## 6. Estabilidad

Candidatas: 8 mejores por score_forma, 4 mejores por score_nivel y la mejor por score_forma de
cada embedding. Se reajustan con 3 muestras nuevas (preprocesamiento, embedding y clustering)
y se compara con el ARI sobre la muestra eval.

In [ ]:
v = res[validas]
mejores_emb = v.loc[v.groupby("embedding")["score_forma"].idxmax()]
candidatas = pd.concat([v.nlargest(8, "score_forma"), v.nlargest(4, "score_nivel"), mejores_emb])
candidatas = candidatas.drop_duplicates("id").reset_index(drop=True)


def reajustar(cid, seed):
    conjunto, prep, emb, eprm, algo, prm = CONFIGS[cid]
    idx_f, _ = muestra_ajuste(seed)
    X = celdas[CONJUNTOS[conjunto]].to_numpy(dtype=float)
    tr = (StandardScaler() if prep == "estándar"
          else make_pipeline(StandardScaler(), PCA(n_components=0.9, random_state=seed)))
    Z_f = tr.fit(X[idx_f]).transform(X[idx_f])
    E_s = None if emb == "ninguno" else embeber(emb, eprm, Z_f[:N_SUB], seed)
    modelo = ajustar(emb, algo, prm, Z_f, Z_f[:N_SUB], E_s, seed, {})
    return modelo.predict(tr.transform(X[idx_eval]))


aris = []
for cid in tqdm(candidatas["id"], desc="bootstrap"):
    a = [adjusted_rand_score(ETIQUETAS[cid], reajustar(cid, 1000 + b)) for b in range(N_BOOT)]
    aris.append({"id": cid, "ari_media": np.mean(a), "ari_min": np.min(a)})
candidatas = candidatas.merge(pd.DataFrame(aris), on="id")
candidatas[COLS_TABLA + ["ari_media", "ari_min"]].round(3)

### 6.1 Finalistas

Entre las estables (ARI medio de al menos 0.8): las 2 mejores por score_forma y la mejor por
score_nivel.

In [ ]:
estables = candidatas[candidatas["ari_media"] >= ARI_ESTABLE]
print(f"Estables: {len(estables)} de {len(candidatas)}")
fin_forma = estables.nlargest(2, "score_forma")
fin_nivel = estables[~estables["id"].isin(fin_forma["id"])].nlargest(1, "score_nivel")
finalistas = pd.concat([fin_forma.assign(lente="forma"), fin_nivel.assign(lente="nivel")])
finalistas[["lente"] + COLS_TABLA + ["ari_media"]].round(3)

## 7. Modelos finales

Las finalistas se reajustan para etiquetar todas las celdas. KMeans sin embedding usa todas
las celdas con `sample_weight` y `n_init=20`; GMM sin embedding, 200k filas ponderadas; las
demás, la muestra sub y KNN. Se agregan como referencia el modelo del notebook 04 y el
ganador del 05 (C_trayectoria, PCA, KMeans k=4).

In [ ]:
def ajustar_final(conjunto, prep, emb, eprm, algo, prm):
    X = celdas[CONJUNTOS[conjunto]].to_numpy(dtype=float)
    tr = (StandardScaler() if prep == "estándar"
          else make_pipeline(StandardScaler(), PCA(n_components=0.9, random_state=RANDOM_STATE)))
    Z_all = tr.fit(X[idx_fit]).transform(X)
    if emb == "ninguno" and algo == "kmeans":
        modelo = ajustar(emb, algo, prm, Z_all, None, None, RANDOM_STATE, {},
                         w=celdas["peso"].to_numpy(), n_init=20)
    elif emb == "ninguno" and algo == "gmm":
        idx_big, _ = muestra_ajuste(RANDOM_STATE, n=N_FINAL)
        modelo = ajustar(emb, algo, prm, Z_all[idx_big], None, None, RANDOM_STATE, {}, n_init=5)
    else:
        Z_sub = Z_all[idx_sub]
        E_sub = None if emb == "ninguno" else embeber(emb, eprm, Z_sub, RANDOM_STATE)
        modelo = ajustar(emb, algo, prm, None, Z_sub, E_sub, RANDOM_STATE, {})
    return modelo.predict(Z_all), Z_all


modelos_finales = {
    "base_04": ("A_base", "estándar", "ninguno", {}, "kmeans", {"k": 4}),
    "ganador_05": ("C_trayectoria", "estandar_pca", "ninguno", {}, "kmeans", {"k": 4}),
}
for i, r in enumerate(finalistas.itertuples(), start=1):
    modelos_finales[f"{r.lente}_{i}"] = CONFIGS[r.id]

etiquetas_finales, metricas_finales = {}, []
for nombre, cfg in tqdm(modelos_finales.items(), desc="modelos finales"):
    labels, Z_all = ajustar_final(*cfg)
    etiquetas_finales[nombre] = labels
    conjunto, prep, emb, eprm, algo, prm = cfg
    metricas_finales.append({
        "modelo": nombre, "conjunto": conjunto, "prep": prep, "embedding": emb,
        "emb_params": texto(eprm), "algoritmo": algo, "params": texto(prm),
        **evaluar(labels[idx_eval], Z_all[idx_eval]),
    })
metricas_finales = pd.DataFrame(metricas_finales)
metricas_finales.round(3)

### 7.1 Coincidencia entre modelos (ARI)

In [ ]:
nombres = list(etiquetas_finales)
pd.DataFrame(
    [[adjusted_rand_score(etiquetas_finales[a][idx_eval], etiquetas_finales[b][idx_eval])
      for b in nombres] for a in nombres],
    index=nombres, columns=nombres,
).round(3)

### 7.2 Perfiles de cada modelo

Resumen ponderado, ordenado por promedio de participación.

In [ ]:
def resumen_perfiles(labels):
    df = celdas.assign(perfil=labels)
    df = df[df["perfil"] >= 0]
    filas = []
    for p, dp in df.groupby("perfil"):
        w = dp["peso"]
        filas.append({
            "pct_pob": w.sum() / df["peso"].sum() * 100,
            "promedio": np.average(dp["promedio"], weights=w),
            "brecha_pres": np.average(dp["brecha_pres"], weights=w),
            "tendencia": np.average(dp["tendencia"], weights=w),
            "desviación": np.average(dp["desviación"], weights=w),
            "P_PJ2025": np.average(dp["P_PJ2025"], weights=w),
            "pct_mujer": np.average(dp["MUJER"], weights=w) * 100,
            "pct_rural_mixta": np.average(dp["TIPOSEC"].astype(str) != "U", weights=w) * 100,
            "edad_dominante": dp.groupby("GRUPO_EDAD")["peso"].sum().idxmax(),
        })
    return pd.DataFrame(filas).sort_values("promedio").reset_index(drop=True).round(1)


for nombre, labels in etiquetas_finales.items():
    conjunto, prep, emb, eprm, algo, prm = modelos_finales[nombre]
    print(f"\n{nombre}: {conjunto} | {prep} | {emb} {texto(eprm)} | {algo} {texto(prm)}")
    print(resumen_perfiles(labels).to_string())

### 7.3 Trayectorias de los perfiles de forma

Trayectoria ponderada de cada perfil de los finalistas de forma, junto al modelo del notebook 04.

In [ ]:
anios = [2009, 2012, 2015, 2018, 2021, 2024]
graficar = ["base_04"] + [n for n in etiquetas_finales if n.startswith("forma")]
fig, axes = plt.subplots(1, len(graficar), figsize=(5 * len(graficar), 4), sharey=True)
for ax, nombre in zip(np.atleast_1d(axes), graficar):
    df = celdas.assign(perfil=etiquetas_finales[nombre])
    for p, dp in df[df["perfil"] >= 0].groupby("perfil"):
        tray = [np.average(dp[e], weights=dp["peso"]) for e in ELECCIONES]
        share = dp["peso"].sum() / df["peso"].sum() * 100
        ax.plot(anios, tray, "o-", label=f"{p} ({share:.0f}%)")
    ax.set_title(nombre)
    ax.set_xticks(anios)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("% participación (ponderada)")
plt.tight_layout()
plt.show()

## 8. Exportacion

In [ ]:
res.to_csv(DIR_PREPROCESSED / "experimentos_forma_embeddings.csv", index=False)
candidatas.to_csv(DIR_PREPROCESSED / "experimentos_forma_estabilidad.csv", index=False)
metricas_finales.to_csv(DIR_PREPROCESSED / "experimentos_forma_finalistas.csv", index=False)

salida = celdas[["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD", "peso"]].copy()
for nombre, labels in etiquetas_finales.items():
    salida[nombre] = labels.astype("int8")
salida.to_parquet(DIR_PREPROCESSED / "celdas_perfiles_candidatos_forma.parquet", index=False)

for f in ["experimentos_forma_embeddings.csv", "experimentos_forma_estabilidad.csv",
          "experimentos_forma_finalistas.csv", "celdas_perfiles_candidatos_forma.parquet"]:
    print(f"{f}: {(DIR_PREPROCESSED / f).stat().st_size / 1024:.0f} KB")